# 01.03 — What Is Cardinality?

This notebook answers four questions:

1. **What** is cardinality in a graph model?
2. **Why** do property graph databases not enforce it?
3. **What** does Orthograph use cardinality for?
4. **What** is cardinality *not* for?

It uses only simple, constant cardinality (`CardinalitySpec(min, max)`).  
Cardinality that varies by node-property values is covered in **01.05 — Cardinality That Depends on Node Properties**.

## 1. What cardinality means

Cardinality is a **count constraint**: it says how many times a node may participate
in a relationship of a given type.

It is expressed as a range `min..max` (standard UML/ER notation):

| Notation | Meaning | Orthograph spec |
|----------|---------|-----------------|
| `0..1`   | at most one | `CardinalitySpec(min=0, max=1)` |
| `1..1`   | exactly one | `CardinalitySpec(min=1, max=1)` |
| `0..*`   | zero or more (no upper bound) | `CardinalitySpec(min=0, max=None)` |
| `1..*`   | at least one (no upper bound) | `CardinalitySpec(min=1, max=None)` |
| `m..n`   | between m and n | `CardinalitySpec(min=m, max=n)` |

In a filmography domain, natural examples are:

- A `Movie` is directed by **exactly one** `Person` — `DIRECTED` source cardinality `1..1`.
- A `Person` may act in **zero or more** movies — `ACTED_IN` source cardinality `0..*`.
- A `Movie` must have **at least one** cast member — `ACTED_IN` target cardinality `1..*`.
- A `Person` lives in **at most one** city — `LIVES_IN` source cardinality `0..1`.

Cardinality is always declared **on the relationship type**, not on the node.  
A relationship has two sides:
- `__source_cardinality__` — how many outgoing edges of this type each **source** node may have.
- `__target_cardinality__` — how many incoming edges of this type each **target** node may have.

The two sides are independent constraints.

In [ ]:
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import (
    CardinalitySpec,
    RelationshipModel,
)
from orthograph.graph_definition.validation import GraphValidator

In [ ]:
from shared.filmography import Movie, Person

In [ ]:
# Declare the filmography relationships with explicit cardinality.
#
# Person and Movie node types come from shared/filmography.py.
# New relationship classes are declared here so cardinality is visible.


class DirectedBy(RelationshipModel):
    """A movie is directed by exactly one person."""

    __label__ = "DIRECTED_BY"
    __source_label__ = "Movie"
    __target_label__ = "Person"
    # Each Movie has exactly one outgoing DIRECTED_BY edge.
    __source_cardinality__ = CardinalitySpec(min=1, max=1)
    # A Person may direct zero or more movies.
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


class ActedIn(RelationshipModel):
    """A person acts in zero or more movies; each movie has at least one cast member."""

    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    # A Person may act in zero or more movies.
    __source_cardinality__ = CardinalitySpec(min=0, max=None)
    # Each Movie must have at least one actor.
    __target_cardinality__ = CardinalitySpec(min=1, max=None)
    role: str


model = GraphDefinition(
    name="Filmography",
    node_types=[Movie, Person],
    relationship_types=[DirectedBy, ActedIn],
)

print("Model assembled.")
print(f"  Relationship types: {model.relationship_labels}")
for rt in model.relationship_types:
    src_max = (
        "*" if rt.__source_cardinality__.max is None else rt.__source_cardinality__.max
    )
    tgt_max = (
        "*" if rt.__target_cardinality__.max is None else rt.__target_cardinality__.max
    )
    print(
        f"  {rt.__source_label__} --[{rt.__label__}]--> {rt.__target_label__}"
        f"   source: {rt.__source_cardinality__.min}..{src_max}"
        f"   target: {rt.__target_cardinality__.min}..{tgt_max}"
    )

## 2. Two faces of cardinality: declared intent and observed reality

Cardinality lives in two distinct places in an Orthograph workflow:

```
Declared intent                                 Observed reality
────────────────                        ────────────────────────────────────
GraphDefinition    ←──── compare() ────►  GraphProfile
(your schema)                             (profiled from a live DB)
```

**Declared intent** is what you write in Python or YAML: the `CardinalitySpec`
attached to a `RelationshipModel`.  It is the designer's *expectation*.

**Observed reality** is what Orthograph *measures* when it profiles a live database:
the minimum, maximum, and average number of edges of each relationship type per
source node.  This is stored in `CardinalityStats` on a `RelationshipTypeProfile`.

`compare(definition, profile)` reconciles the two sides and surfaces drift:
if the observed minimum degree falls outside the declared bounds, it reports a
`CARDINALITY_VIOLATION`.

> Live-database profiling requires a backend connection.  
> See **05.01 — Profile vs. Definition** for a runnable profiling example.

## 3. Why property graph databases do not enforce relationship cardinality

Neo4j, Memgraph, and other property graph databases enforce:

- **Uniqueness constraints** — at most one node with a given property value.
- **Existence constraints** — a property must be present on every node of a label.

They do **not** enforce relationship cardinality.  There is no native database
mechanism that prevents a `Movie` from having two `DIRECTED_BY` edges, or zero.
The count is assumed by the application but never checked by the database.

This is the gap Orthograph fills.  The declared `GraphDefinition` is the
application's source of truth for *intended* cardinality.  Orthograph checks
it — the database does not.

## 4. What Orthograph uses cardinality for: in-memory validation

Given a set of nodes and relationships in memory (for example, data about to
be written to the database), `GraphValidator.validate()` checks each node's
**edge count** against the declared cardinality.

Concretely, for each node:
- It counts how many outgoing edges of each relationship type the node has.
- It counts how many incoming edges of each relationship type the node has.
- It checks each count against the relevant `CardinalitySpec.contains(count)`.

It counts **relationship degree** — not node properties.

In [ ]:
# --- Valid data ---
# Inception: directed by exactly 1 person, acted in by 2 people.

nodes = [
    {"__label__": "Movie", "title": "Inception", "released": 2010},
    {"__label__": "Person", "name": "Christopher Nolan"},
    {"__label__": "Person", "name": "Leonardo DiCaprio"},
    {"__label__": "Person", "name": "Ken Watanabe"},
]

relationships = [
    # One DIRECTED_BY — satisfies source cardinality 1..1 on Movie
    {
        "__label__": "DIRECTED_BY",
        "__source_uid__": "Inception",
        "__target_uid__": "Christopher Nolan",
    },
    # Two ACTED_IN — satisfies target cardinality 1..* on Movie
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Leonardo DiCaprio",
        "__target_uid__": "Inception",
        "role": "Cobb",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Ken Watanabe",
        "__target_uid__": "Inception",
        "role": "Saito",
    },
]

validator = GraphValidator(model)
result = validator.validate(nodes, relationships)

print("Valid data:")
print(f"  is_valid: {result.is_valid}")
print(f"  issues:   {len(result.issues)}")

In [ ]:
# --- Violation: a Movie with two directors (source cardinality is ONE) ---

nodes_two_directors = [
    {"__label__": "Movie", "title": "Inception", "released": 2010},
    {"__label__": "Person", "name": "Christopher Nolan"},
    {"__label__": "Person", "name": "Impostor"},
    {"__label__": "Person", "name": "Leonardo DiCaprio"},
]

relationships_two_directors = [
    {
        "__label__": "DIRECTED_BY",
        "__source_uid__": "Inception",
        "__target_uid__": "Christopher Nolan",
    },
    {
        "__label__": "DIRECTED_BY",
        "__source_uid__": "Inception",
        "__target_uid__": "Impostor",
    },
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Leonardo DiCaprio",
        "__target_uid__": "Inception",
        "role": "Cobb",
    },
]

result_two = validator.validate(nodes_two_directors, relationships_two_directors)

print("Two directors (violates source cardinality 1..1):")
print(f"  is_valid: {result_two.is_valid}")
for issue in result_two.errors:
    print(f"  [{issue.code}] {issue.message}")

In [ ]:
# --- Violation: a Movie with no director (source cardinality is ONE, min=1) ---

nodes_no_director = [
    {"__label__": "Movie", "title": "Inception", "released": 2010},
    {"__label__": "Person", "name": "Leonardo DiCaprio"},
]

relationships_no_director = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Leonardo DiCaprio",
        "__target_uid__": "Inception",
        "role": "Cobb",
    },
]

result_none = validator.validate(nodes_no_director, relationships_no_director)

print("No director (violates source cardinality 1..1, min=1):")
print(f"  is_valid: {result_none.is_valid}")
for issue in result_none.errors:
    print(f"  [{issue.code}] {issue.message}")

In [ ]:
# --- Violation: a Movie with no actors (target cardinality is 1..*, min=1) ---

nodes_no_actors = [
    {"__label__": "Movie", "title": "Inception", "released": 2010},
    {"__label__": "Person", "name": "Christopher Nolan"},
]

relationships_no_actors = [
    {
        "__label__": "DIRECTED_BY",
        "__source_uid__": "Inception",
        "__target_uid__": "Christopher Nolan",
    },
]

result_no_cast = validator.validate(nodes_no_actors, relationships_no_actors)

print("No actors (violates target cardinality 1..*, min=1):")
print(f"  is_valid: {result_no_cast.is_valid}")
for issue in result_no_cast.errors:
    print(f"  [{issue.code}] {issue.message}")

In [ ]:
# --- Custom CardinalitySpec: a festival requires 2 to 5 films ---


from orthograph.graph_definition.models import CardinalitySpec, NodeModel


class Festival(NodeModel):
    __label__ = "Festival"
    __uid_field__ = "name"
    name: str


class FestivalMovie(NodeModel):
    __label__ = "FestivalMovie"
    __uid_field__ = "title"
    title: str


class Screens(RelationshipModel):
    """A festival screens between 2 and 5 films."""

    __label__ = "SCREENS"
    __source_label__ = "Festival"
    __target_label__ = "FestivalMovie"
    # Each Festival screens between 2 and 5 films.
    __source_cardinality__ = CardinalitySpec(min=2, max=5)
    __target_cardinality__ = CardinalitySpec(min=0, max=None)


festival_model = GraphDefinition(
    name="Festival",
    node_types=[Festival, FestivalMovie],
    relationship_types=[Screens],
)
v_festival = GraphValidator(festival_model)

festival_nodes = [
    {"__label__": "Festival", "name": "Cannes"},
    {"__label__": "FestivalMovie", "title": "Film A"},
    {"__label__": "FestivalMovie", "title": "Film B"},
    {"__label__": "FestivalMovie", "title": "Film C"},
]

# 3 films — within 2..5
r_ok = v_festival.validate(
    festival_nodes,
    [
        {
            "__label__": "SCREENS",
            "__source_uid__": "Cannes",
            "__target_uid__": "Film A",
        },
        {
            "__label__": "SCREENS",
            "__source_uid__": "Cannes",
            "__target_uid__": "Film B",
        },
        {
            "__label__": "SCREENS",
            "__source_uid__": "Cannes",
            "__target_uid__": "Film C",
        },
    ],
)
print(f"3 films (valid 2..5):  is_valid={r_ok.is_valid}")

# 1 film — below min=2
r_low = v_festival.validate(
    festival_nodes,
    [{"__label__": "SCREENS", "__source_uid__": "Cannes", "__target_uid__": "Film A"}],
)
print(f"1 film  (violates min=2): is_valid={r_low.is_valid}")
for issue in r_low.errors:
    print(f"  [{issue.code}] {issue.message}")

## 5. What Orthograph uses cardinality for: profiling a live database

When Orthograph inspects a live database, it measures the actual edge counts
per relationship type and stores them in `CardinalityStats`
(`min`, `max`, `mean`).  `compare(definition, profile)`
then checks the observed minimum degree against the declared `CardinalitySpec`.

If the observed `min` falls outside `[spec.min, spec.max]`, a
`CARDINALITY_VIOLATION` finding is emitted.

This means a declared cardinality serves a **dual purpose**:

1. It validates **in-memory data** before it is written to the database.
2. It provides a **reference point** against which a live database can be profiled
   and drift detected.

> Runnable profiling examples require a backend connection.  
> See **05.01 — Profile vs. Definition** for a worked example.

## 6. What cardinality is NOT for: Cypher query validation

Orthograph validates Cypher query strings against the declared schema: it checks
that labels, relationship types, property names, and endpoints referenced in a
query exist in the `GraphDefinition`.

Cardinality is **not checked during query validation**, for a structural reason:
a Cypher query string is a *pattern*, not a count.  It describes which nodes
and relationships to match, but contains no information about how many edges
a node will have.  That count only exists at runtime, across actual data.

Cardinality validation belongs to **data validation** (before write) and
**profiling** (against a live database), not to static query analysis.

## 7. The two orthogonal axes: optionality vs. cardinality

A common source of confusion: `0..*` means *each node may have zero
edges of this type* — but it does **not** mean *the relationship type is absent*.

Two independent axes control this:

| Axis | Question | Mechanism |
|------|----------|-----------|
| **Entity-level optionality** | Must this relationship *type* appear at all? | `__optional__ = True/False` |
| **Cardinality** | How many edges of this type may each *node* have? | `CardinalitySpec(min, max)` |

`0..*` with `__optional__ = False` means: *the relationship type must
appear in the graph, but individual nodes are not required to participate.*

`1..*` with `__optional__ = False` means: *the type must appear, and
every node must have at least one edge of this type.*

They are orthogonal: set them independently based on what your domain requires.

> **01.04 — Optionality and Cardinality** covers all three optionality levels
> (property, entity, cardinality) and shows how they combine in practice.